# Anomaly-Based Network Intrusion Detection System
## Notebook 03: Feature Engineering

Creates new features, reduces dimensionality, and selects the most
informative features before model training.

## 1. Imports

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA

from src.features.engineer import FeatureEngineer

print('Ready')

## 2. Load Processed Data

In [ ]:
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

X_train = pd.read_csv(f'{DATA_DIR}/X_train.csv')
X_val   = pd.read_csv(f'{DATA_DIR}/X_val.csv')
X_test  = pd.read_csv(f'{DATA_DIR}/X_test.csv')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv').squeeze()
y_val   = pd.read_csv(f'{DATA_DIR}/y_val.csv').squeeze()
y_test  = pd.read_csv(f'{DATA_DIR}/y_test.csv').squeeze()

print(f'Train {X_train.shape}  Val {X_val.shape}  Test {X_test.shape}')

## 3. Mutual-Information Feature Selection

In [ ]:
selector = SelectKBest(score_func=mutual_info_classif, k=30)
selector.fit(X_train, y_train)

scores = pd.Series(selector.scores_, index=X_train.columns)
top30 = scores.nlargest(30)

plt.figure(figsize=(10, 8))
top30.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top-30 Features by Mutual Information', fontsize=14)
plt.xlabel('MI Score')
plt.tight_layout()
plt.show()

SELECTED_FEATURES = top30.index.tolist()
print(f'Selected {len(SELECTED_FEATURES)} features')

## 4. PCA Visualisation (2-D)

In [ ]:
from sklearn.preprocessing import LabelEncoder

pca = PCA(n_components=2, random_state=42)
sample_idx = X_train.sample(5000, random_state=42).index
X_pca = pca.fit_transform(X_train.loc[sample_idx, SELECTED_FEATURES])
y_sample = y_train.loc[sample_idx]

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                      c=y_sample, cmap='bwr', alpha=0.4, s=8)
plt.colorbar(scatter, label='Label (0=Normal, 1=Attack)')
plt.title('PCA 2-D projection (sample of 5 000)', fontsize=13)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

## 5. Save Feature Metadata

In [ ]:
feat_dir = os.path.join(PROJECT_ROOT, 'data', 'features')
os.makedirs(feat_dir, exist_ok=True)

joblib.dump(SELECTED_FEATURES, f'{feat_dir}/selected_features.pkl')
joblib.dump(selector,          f'{feat_dir}/mi_selector.pkl')

print(f'Saved feature metadata to {feat_dir}')
print('Next: 04_ml_models.ipynb')